In [1]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import pandas as pd

# Import du processeur de production spécialisé
from tools.OI_class_OP import OI_ProductionProcessor
from tools.OI_Dashboard import ProductionDashboard
from tools.OI_Dashboard_v2 import AjouterVisualisationsAvancees


# Configuration de l'affichage pour voir toutes les colonnes
pd.set_option('display.max_columns', None)

In [2]:
# Cellule 2 : Définition des métadonnées de tags API
tags = [
    {'tag':'WQ33222VA', 'nom':'ester_cons','info':'Totalisation du Peson Acetate/Propionate' },
    {'tag':'NOP_ESTERS', 'nom':'ester_nop','info':"Nombres des opérations d'esters" },
    {'tag':'3340_type', 'nom':'A/P','info':'Acetate ou Propionate' },
    {'tag':'CTY_ACV43A_Teneur Vit. A (UV)', 'nom':'Acetate_UV','info':'ACV43A teneur en acetate'},
    {'tag':'CTY_A3340FGB_Teneur arr. Vit. A (UV)', 'nom':'Propionate_UV', 'info':'A3340FGB teneur en propionate'},
    {'tag':'PU3310VA_Sign','nom':'PU3310','info':'signature du PU3310'},
    {'tag':'PU3320VA_Sign','nom':'PU3320','info':'signature du PU3320'},
    {'tag':'PU3340VA_Sign','nom':'PU3340','info':'signature du PU3340'},
    {'tag':'FQ32202VA_UV','nom':'Hexane','info':'VA diluée dans de l\'hexane'},
    {'tag':'CTY_A3230A_Teneur en rétinol', 'nom':'Retinol_UV','info':'A3230A teneur en rétinol lavé'},
    {'tag':'LI33203VA','nom':'R33020','info':'niveau du R33020'},
    {'tag':'LI33218VA','nom':'R33022','info':'niveau du R33022'},
    {'tag':'LI33225VA','nom':'R33061','info':'niveau du R33061'},
    {'tag':'WI33222VA','nom':'R33060','info':'peson du R33060'},
    {'tag':'FQ32202VA','nom':'retinol_cons','info':'peson du R33060'},
    {'tag':'LI32209VA','nom':'R32031','info':'niveau du R32031'},
]

In [3]:
# Cellule 3 : Variable Produits unifiée (Acetate & Propionate)
# Plus aucune distinction batch / continu pour le calcul global du stock d'un produit.
produits = [
    {
        'nom': 'Ester',
        'conso': {
            'value': 'ester_cons',
            'scale': 1e-9,
            'type':'A/P',
            'uv': ['Acetate_UV', 'Propionate_UV'],
        },
        'CMJ': 9.7,
        'NOP': {
            'value':'NOP_ESTERS',
            'scale': 1,
            'type': None,
        },
        'stock': [
            # Batchs
            {'pu': 'PU3310', 'in': 540, 'out': 710, 'value': 'Hexane', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
            {'pu': 'PU3310', 'in': 710, 'out': 2320, 'value': None, 'uv': None, 'scale': 2.71},
            {'pu': 'PU3320', 'in': 430, 'out': 2020, 'value': None, 'uv': None, 'scale': 2.71},
            # Continus
            {'pu': None, 'value': 'R33020', 'min': 14, 'epalage': [[28.62, 21.22, 3.866, -0.0983], [-228.84, 74.557]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33022', 'min': 14, 'epalage': [[6.25, 5.4525, 0.898, -0.0256], [-35, 97, 16.039]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33061', 'min': 18, 'epalage': [[0.69, 0.72, 0.296, -0.0059], [-30.03, 5.836]], 'uv': None, 'scale': 0.95 , 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': 'PU3340', 'in': 0, 'out': 710, 'value': 'R33060', 'uv': None, 'scale': 0.95, 'cond': 'A/P', 'val_cond': [1/344, 1/359]}
        ]
    },
    {
        'nom': 'Retinol',
        'conso': {
            'value':'retinol_cons',
            'scale': 1e-5, # Ajustement de l'échelle pour le retinol g et analyse en %
            'uv': ['Retinol_UV','Retinol_UV'],
        },
        'CMJ': 1,
        'stock': [
            # Batchs
            {'pu': None, 'value': 'R32031', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
        ]
    }
]

In [4]:
# Cellule 4 : Initialisation du processeur de production spécialisé
processor = OI_ProductionProcessor(
    url_base = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    start = '2026-01-01',
    end = '2026-12-31',
    tags_metadata = tags,
    produits = produits,
    interval = 'PT20M',
    verbose = False
)

In [5]:
# Cellule 5 : Téléchargement et calcul automatique des bilans par produit
processor.merge()
processor.compute_production_balance()

# Visualisation des premières lignes calculées
processor.data.describe()

,ester_cons,ester_nop,A/P,Acetate_UV,Propionate_UV,PU3310,PU3320,PU3340,Hexane,Retinol_UV,R33020,R33022,R33061,R33060,retinol_cons,R32031,consommation_Ester,stock_Ester,consommation_Retinol,stock_Retinol,conso_delta_Ester,delta_stock_Ester,production_Ester,conso_delta_Retinol,delta_stock_Retinol,production_Retinol
count,1.089700e+04,10897.000000,10897.000000,1.089700e+04,1.089700e+04,10894.000000,10895.000000,10893.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000,10897.000000
mean,2.449690e+06,1960.896296,0.076307,2.226211e+06,2.281373e+06,1013.200961,941.372673,461.909886,2564.116008,20.789453,45.595810,4.978748,7.202522,685.783090,462338.216370,47.655698,508.947208,6.660529,391537.636363,0.012008,508.947208,0.433117,509.380325,391537.636363,-0.002162,391537.634202
std,1.429036e+05,117.163928,0.265373,3.860027e+04,5.114101e+04,598.357505,529.165418,111.572199,1861.579249,0.370188,20.342066,2.410457,10.976399,344.939887,251312.519679,20.447020,319.752895,2.248667,486758.589099,0.005155,319.752895,2.248667,320.293107,486758.589099,0.005155,486758.589379
min,2.221310e+06,1774.000000,0.000000,2.078915e+06,2.188000e+06,100.000000,100.000000,313.116667,0.000000,20.000000,-7.918751,-0.695313,-0.337315,6.278194,640.456000,0.220855,0.000000,0.724677,0.000000,0.000056,0.000000,-5.502735,-1.516407,0.000000,-0.014113,0.000000
25%,2.314830e+06,1850.706353,0.000000,2.221983e+06,2.242000e+06,400.000000,400.000000,410.000000,0.000000,20.600000,24.041651,3.619748,2.211112,426.962575,247246.000000,39.761114,207.463864,5.528258,64.687337,0.010008,207.463864,-0.699155,207.908419,64.687337,-0.004162,64.690968
50%,2.433800e+06,1947.000000,0.000000,2.231260e+06,2.314000e+06,1310.000000,1113.675000,410.000000,3931.900000,20.800000,44.906267,5.949938,2.954918,683.419807,429755.000000,45.012744,471.567857,7.029563,149.030636,0.011510,471.567857,0.802151,473.779116,149.030636,-0.002659,149.035069
75%,2.575120e+06,2063.000000,0.000000,2.245049e+06,2.314000e+06,1310.000000,1297.177083,410.000000,4036.650000,21.200000,64.402751,6.512820,4.125066,940.895295,666099.000000,66.813987,789.448476,8.169719,996792.166644,0.016763,789.448476,1.942306,788.687741,996792.166644,0.002594,996792.169002
max,2.712010e+06,2177.000000,1.000000,2.325113e+06,2.626000e+06,2320.033333,2249.987500,810.000000,4836.070000,21.200000,91.504357,12.239069,68.457970,1321.036235,997389.108434,79.734400,1097.996182,12.528535,996886.627773,0.019909,1097.996182,6.301123,1100.358127,996886.627773,0.005740,996886.627584


In [6]:
# ✨ INITIALISER LE DASHBOARD ✨
dashboard = ProductionDashboard(processor)
AjouterVisualisationsAvancees(dashboard) 
 
print("\n✓ Dashboard prêt pour utilisation!")


✓ Dashboard initialisé
  Produits: Ester, Retinol
  Période: 2026-01-01 → 2026-06-01
✅ Visualisations avancées ajoutées au dashboard!

   Nouvelles méthodes disponibles:
   • dashboard.plot_histogramme_tous_produits(mois=3)
   • dashboard.plot_waterfall_mois(mois=3)
   • dashboard.plot_histogramme_jours_mois_v1(mois=3, nom_produit='Ester')
   • dashboard.plot_histogramme_jours_mois_v2(mois=3, nom_produit='Ester')


✓ Dashboard prêt pour utilisation!


In [7]:
# Afficher le résumé complet
dashboard.resume_complet()



  RÉSUMÉ COMPLET DU DASHBOARD

📅 Période de données: 2026-01-01 → 2026-06-01
📦 Produits disponibles: Ester, Retinol

  EXEMPLE D'UTILISATION

# Afficher les KPIs d'un jour
dashboard.afficher_kpis_journaliers(jour=15, mois=3, annee=2024)

# Afficher les KPIs d'un mois
dashboard.afficher_kpis_mensuels(mois=3, annee=2024)

# Voir le bilan graphique d'une journée
dashboard.afficher_bilan_journalier(jour=15, mois=3)

# Voir l'évolution d'un produit
dashboard.afficher_bilan_produit('Acetate')

# Comparer les N derniers jours
dashboard.afficher_comparaison_jours(num_jours=7)
        



In [8]:
# Voir l'évolution temporelle d'un produit
dashboard.afficher_bilan_produit('Ester')


In [9]:
dashboard.afficher_kpis_mensuels(mois=5,annee=2026)

____________________________________________________________
BILAN MENSUEL - MAI 2026 (Terminé)
Période : du 01/05/2026 à 02:00 au 01/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 263.8927
  Variation de Stock     : -2.5546
  Production             : 261.3381
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 79.0581
  Variation de Stock     : 0.0060
  Production             : 79.0641
____________________________________________________________
____________________________________________________________

  KPIS MENSUELS - MAY 2026
  Production Mensuelle           :       340.40 unités         
  Consommation Mensuelle         :       342.95 unités         
  Variation Stock                :        -2.55 unités         



[KPI(nom='Production Mensuelle', valeur=340.402216068557, unite='unités', variation=None, couleur='#4CAF50'),
 KPI(nom='Consommation Mensuelle', valeur=342.9508231728803, unite='unités', variation=None, couleur='#FF9800'),
 KPI(nom='Variation Stock', valeur=-2.5486071043767557, unite='unités', variation=None, couleur='#F44336')]

In [10]:
dashboard.afficher_kpis_journaliers(jour=31, mois=5, annee=2026)

____________________________________________________________
BILAN JOURNALIER - 31 MAI 2026 (Terminé)
Période : du 31/05/2026 à 02:00 au 01/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.3226
  Variation de Stock     : -3.5124
  Production             : 4.8102
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 1.6025
  Variation de Stock     : 0.0002
  Production             : 1.6027
____________________________________________________________
____________________________________________________________

  KPIS JOURNALIERS - 31/5/2026
  Production Totale              :         6.41 unités         
  Consommation                   :         9.93 unités         
  Variation Stock                :        -3.51 unités         



[KPI(nom='Production Totale', valeur=6.412865755951316, unite='unités', variation=None, couleur='#4CAF50'),
 KPI(nom='Consommation', valeur=9.925073800071914, unite='unités', variation=None, couleur='#FF9800'),
 KPI(nom='Variation Stock', valeur=-3.5122080441129637, unite='unités', variation=None, couleur='#F44336')]

In [11]:
dashboard.plot_histogramme_tous_produits(mois=5)


____________________________________________________________
BILAN MENSUEL - MAI 2026 (Terminé)
Période : du 01/05/2026 à 02:00 au 01/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 263.8927
  Variation de Stock     : -2.5546
  Production             : 261.3381
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 79.0581
  Variation de Stock     : 0.0060
  Production             : 79.0641
____________________________________________________________
____________________________________________________________



  📋 RÉSUMÉ - HISTOGRAMME MAY 2026
Produit              Production         Consommation       Var. Stock         Efficacité     
-----------------------------------------------------------------------------------------------
Ester                261.34             263.89             -2.55              99.0           %
Retinol              79.06              79.06              0.01               100.0          %
-----------------------------------------------------------------------------------------------
TOTAL                340.40             342.95             -2.55             



In [12]:
dashboard.afficher_kpis_mensuels(mois=5, annee=2026)

____________________________________________________________
BILAN MENSUEL - MAI 2026 (Terminé)
Période : du 01/05/2026 à 02:00 au 01/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 263.8927
  Variation de Stock     : -2.5546
  Production             : 261.3381
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 79.0581
  Variation de Stock     : 0.0060
  Production             : 79.0641
____________________________________________________________
____________________________________________________________

  KPIS MENSUELS - MAY 2026
  Production Mensuelle           :       340.40 unités         
  Consommation Mensuelle         :       342.95 unités         
  Variation Stock                :        -2.55 unités         



[KPI(nom='Production Mensuelle', valeur=340.402216068557, unite='unités', variation=None, couleur='#4CAF50'),
 KPI(nom='Consommation Mensuelle', valeur=342.9508231728803, unite='unités', variation=None, couleur='#FF9800'),
 KPI(nom='Variation Stock', valeur=-2.5486071043767557, unite='unités', variation=None, couleur='#F44336')]

In [13]:
dashboard.plot_histogramme_jours_mois_v2(mois=5, annea=2026, nom_produit='Ester')

____________________________________________________________
BILAN JOURNALIER - 01 MAI 2026 (Terminé)
Période : du 01/05/2026 à 02:00 au 02/05/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 11.1320
  Variation de Stock     : 0.0114
  Production             : 11.1434
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 3.4278
  Variation de Stock     : 0.0075
  Production             : 3.4354
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 MAI 2026 (Terminé)
Période : du 02/05/2026 à 02:00 au 03/05/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
-----


  📊 STATISTIQUES - PRODUCTION ESTER PAR JOUR - MAY 2026
Nombre de jours complets:           31
Production moyenne:                 8.43
Production min/max:                 2.65 / 12.14
Écart-type:                         2.27
Coefficient de variation:           27.0%
Production totale mois:             261.34
TRS mois: (jours complets)          8.43
Jours au-dessus de la moyenne:      16 / 31

🎯 RATIO OOE (Production / CMJ en %):
  CMJ (Cible Journalière):            9.70
  OOE Moyen (par jour):               86.9%
  OOE Min/Max (par jour):             27.3% / 125.1%
  OOE Cumulé à date:                  86.9%
  Jours > 100%:                       8 / 31

  Détail OOE par jour:
    J01:  114.9% ✅
    J02:   86.4% ⚠️ 
    J03:   57.6% ❌
    J04:  125.1% ✅
    J05:   73.5% ❌
    J06:  116.2% ✅
    J07:   90.9% ⚠️ 
    J08:  111.6% ✅
    J09:   86.1% ⚠️ 
    J10:   97.0% ⚠️ 
    J11:   80.7% ⚠️ 
    J12:   56.0% ❌
    J13:   81.5% ⚠️ 
    J14:   61.7% ❌
    J15:   87.5% ⚠️ 
    J16:   90

In [14]:
dashboard.afficher_bilan_journalier(jour=1, mois=6, annee=2026)

____________________________________________________________
BILAN JOURNALIER - 01 JUIN 2026 (À date (En cours))
Période : du 01/06/2026 à 02:00 au 01/06/2026 à 12:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 2.7668
  Variation de Stock     : 2.6550
  Production             : 5.4217
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 1.6935
  Variation de Stock     : -0.0086
  Production             : 1.6849
____________________________________________________________
____________________________________________________________
